# Lesson 30: Interpretability and explainability

*Companion notebook for [the lesson page](../lessons/30-interpretability.html). Worked solutions are in `solutions/30-interpretability-solutions.ipynb`; try each exercise first.*

Find out why a model made a prediction, what it has learned and how far to trust the answer: coefficients, permutation importance and Shapley values for tabular models, gradients for neural networks, and probes, the logit lens, activation patching and sparse autoencoders inside a small transformer.

**Runs on:** Any computer (CPU), offline. The whole notebook runs in about a minute. Two optional cells download the shap library and GPT-2 small.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn torch

## Why explain a model?

A model can score well and still be right for the wrong reasons. Explanations are for debugging, for the people a decision affects, for auditors and for science. Keep in mind throughout: **an explanation is a model of the model**. It describes what the model learned, not how the world works, and it can be wrong about the model too.

## Models you can read

The diabetes data in original units: 442 patients, ten baseline measurements (age, sex, BMI, mean blood pressure `bp`, six blood serum values `s1` to `s6`), and disease progression one year later.

### Linear models: read the coefficients, carefully

With standardised features, a coefficient is the change in prediction per standard deviation, holding the other features fixed. Watch what happens to `s1` and `s2`, which are correlated at 0.9.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge

# 442 patients, 10 baseline measurements; target = disease progression a year later
X, y = load_diabetes(return_X_y=True, as_frame=True, scaled=False)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

ols = make_pipeline(StandardScaler(), LinearRegression()).fit(X_tr, y_tr)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=20)).fit(X_tr, y_tr)
coefs = pd.DataFrame({"ols": ols[-1].coef_, "ridge": ridge[-1].coef_},
                     index=X.columns)
print(coefs.round(1).T)
print("correlation of s1 and s2:", round(X.s1.corr(X.s2), 2))

### Shallow trees and additive models

A generalised additive model (GAM) gives each feature its own curve and adds them up:

$$
f(\mathbf{x}) = \beta_0 + f_1(x_1) + f_2(x_2) + \dots + f_p(x_p)
$$

A spline basis per feature followed by a linear model is exactly that.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.preprocessing import SplineTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(tree, feature_names=list(X.columns)))

# A GAM: one smooth curve per feature (a spline basis), added up by a linear model
gam = make_pipeline(StandardScaler(), SplineTransformer(n_knots=5, degree=3),
                    Ridge(alpha=20))
params = dict(n_estimators=200, max_depth=2, learning_rate=0.05,
              subsample=0.8, random_state=0)
gbm = GradientBoostingRegressor(**params)
for name, model in [("ridge", ridge), ("tree, depth 2", tree),
                    ("GAM", gam), ("boosting", gbm)]:
    r2 = cross_val_score(model, X, y, cv=5, scoring="r2").mean()
    print(f"{name:14s} cross-validated R^2 {r2:.2f}")
gam.fit(X_tr, y_tr)
gbm.fit(X_tr, y_tr)

The GAM's curves are the whole model, so plotting them shows everything it does with each feature.

In [ ]:
import matplotlib.pyplot as plt

# Because the GAM is additive, varying one feature moves the prediction by the
# same amount whatever the other features are: each curve is the whole story.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
base = X_tr.median().to_frame().T
for ax, feature in zip(axes, ["bmi", "bp", "s5"]):
    grid = np.linspace(X[feature].min(), X[feature].max(), 100)
    rows = pd.concat([base] * len(grid), ignore_index=True)
    rows[feature] = grid
    effect = gam.predict(rows)
    ax.plot(grid, effect - effect.mean())
    ax.set_xlabel(feature)
axes[0].set_ylabel("effect on prediction")
fig.suptitle("GAM shape functions")
plt.tight_layout()
plt.show()

## Global explanations for any model

### Impurity importance versus permutation importance

Permutation importance: the drop in held-out score when one feature is shuffled,

$$
I_j = s(f, X, y) - \frac{1}{K}\sum_{k=1}^{K} s\big(f, X^{(k)}_{\text{shuffle } j}, y\big)
$$

We add a column of pure noise and compare it with the trees' built-in impurity importance.

In [ ]:
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(0)
Xn = X.assign(noise=rng.normal(size=len(X)))          # a column of pure noise
Xn_tr, Xn_te = Xn.loc[X_tr.index], Xn.loc[X_te.index]
gbm_noise = GradientBoostingRegressor(**params).fit(Xn_tr, y_tr)

perm = permutation_importance(gbm_noise, Xn_te, y_te, n_repeats=30, random_state=0)
imp = pd.DataFrame({"impurity": gbm_noise.feature_importances_,
                    "permutation": perm.importances_mean,
                    "perm_sd": perm.importances_std}, index=Xn.columns)
print(imp.sort_values("permutation", ascending=False).round(3))

### Partial dependence and ICE curves

$$
\mathrm{PD}_j(v) = \frac{1}{n}\sum_{i=1}^{n} f\big(v, \mathbf{x}_{i,\setminus j}\big)
$$

An ICE curve is the same sweep for one row; the PD curve is their average. For an additive model the centred ICE curves are identical.

In [ ]:
def ice_curves(model, X, feature, grid):
    """One curve per row: the prediction as `feature` sweeps over `grid`."""
    curves = []
    for v in grid:
        X_v = X.copy()
        X_v[feature] = v                  # give every patient the same value
        curves.append(model.predict(X_v))
    return np.array(curves).T             # (rows, grid points)

grid = np.quantile(X_tr.bmi, [0.05, 0.25, 0.5, 0.75, 0.95])
print("bmi grid       ", grid.round(1))
for name, model in [("GAM", gam), ("boosting", gbm)]:
    ice = ice_curves(model, X_te, "bmi", grid)
    centred = ice - ice[:, :1]            # start every curve at zero
    print(f"{name:8s} PD    ", ice.mean(axis=0).round(1))
    print(f"{name:8s} ICE sd", centred.std(axis=0).round(1))

The same thing with scikit-learn's display: thin lines are ICE curves, the thick line is partial dependence.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Thin lines: ICE curves for 50 test patients. Thick line: their average (PD).
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
PartialDependenceDisplay.from_estimator(gbm, X_te, ["bmi", "s5"], kind="both",
                                        subsample=50, random_state=0, ax=axes)
plt.tight_layout()
plt.show()

### The correlated-features trap

Add a near-identical copy of BMI and retrain. The model is as accurate as before; watch what happens to the importance of each copy.

In [ ]:
rng = np.random.default_rng(1)
X_dup = X.assign(bmi_copy=X.bmi + rng.normal(0, 0.5, len(X)))   # near-duplicate
Xd_tr, Xd_te = X_dup.loc[X_tr.index], X_dup.loc[X_te.index]
gbm_dup = GradientBoostingRegressor(**params).fit(Xd_tr, y_tr)

before = permutation_importance(gbm, X_te, y_te, n_repeats=30, random_state=0)
after = permutation_importance(gbm_dup, Xd_te, y_te, n_repeats=30, random_state=0)
print("test R^2 without and with the copy:",
      round(gbm.score(X_te, y_te), 2), round(gbm_dup.score(Xd_te, y_te), 2))
print("importance of bmi, original model: ", before.importances_mean[2].round(3))
print("importance of bmi and bmi_copy:    ", after.importances_mean[[2, 10]].round(3))

### Hands-on: importance for groups of features

1. Write `group_importance(model, X, y, cols)` that shuffles all the columns in `cols` with the same permutation of rows and returns the mean drop in $R^2$ over 30 repeats.
2. Apply it to `["bmi", "bmi_copy"]` for `gbm_dup`, and to `["bmi"]` for the original `gbm`.
3. Apply it to the cholesterol group `["s1", "s2", "s3", "s4"]` of `gbm`, and to each of the four alone.

You are done when the grouped BMI importance is back near the original model's 0.16.

In [ ]:
# Your code here

## Local explanations: why this prediction?

We explain one test patient with a high predicted progression score.

### LIME: a simple model of the model, nearby

Perturb the input, weight the perturbed points by closeness, fit a weighted linear model. Its coefficients are local slopes, here per standard deviation of each feature.

In [ ]:
x = X_te.loc[71]                          # one patient with a high predicted score
print("prediction for this patient:", gbm.predict(x.to_frame().T).round(1))

def lime(model, x, X_ref, n=5000, width=1.0, seed=0):
    """Fit a weighted linear model to the black box in a neighbourhood of x."""
    rng = np.random.default_rng(seed)
    mu, sd = X_ref.mean().to_numpy(), X_ref.std().to_numpy()
    z = (x.to_numpy() - mu) / sd                       # work in standard units
    Z = z + rng.normal(size=(n, len(z)))               # perturb around x
    preds = model.predict(pd.DataFrame(Z * sd + mu, columns=X_ref.columns))
    w = np.exp(-((Z - z) ** 2).sum(1) / (width ** 2 * len(z)))  # near = heavy
    local = Ridge(alpha=1.0).fit(Z - z, preds, sample_weight=w)
    return pd.Series(local.coef_, index=X_ref.columns)  # slope per standard deviation

expl = pd.DataFrame({f"width {w}": lime(gbm, x, X_tr, width=w) for w in [0.5, 1, 2]})
print(expl.round(1).T)

### Shapley values: sharing out the prediction

$$
\phi_j = \sum_{S \subseteq N \setminus \{j\}} \frac{|S|!\,(p-|S|-1)!}{p!}\,\big(v(S \cup \{j\}) - v(S)\big)
$$

where $v(S)$ is the mean prediction when the features in $S$ take this patient's values and the rest come from background patients. Efficiency: the contributions sum to $f(\mathbf{x}) - \mathbb{E}[f(X)]$.

In [ ]:
from itertools import product
from math import factorial

def shapley_exact(model, x, background):
    """Exact Shapley values by enumerating all 2^p coalitions of features."""
    p = len(x)
    coalitions = np.array(list(product([False, True], repeat=p)))   # (2^p, p)
    # v(S): mean prediction with S taken from x and the rest from background rows
    B = background.to_numpy()
    hybrid = np.where(coalitions[:, None, :], x.to_numpy(), B[None, :, :])
    preds = model.predict(pd.DataFrame(hybrid.reshape(-1, p), columns=x.index))
    v = preds.reshape(len(coalitions), len(B)).mean(axis=1)
    index = {tuple(S): k for k, S in enumerate(coalitions)}
    phi = np.zeros(p)
    for S, v_S in zip(coalitions, v):
        s = S.sum()
        for j in np.flatnonzero(~S):                     # every feature not yet in S
            S_j = S.copy()
            S_j[j] = True
            weight = factorial(s) * factorial(p - s - 1) / factorial(p)
            phi[j] += weight * (v[index[tuple(S_j)]] - v_S)
    return pd.Series(phi, index=x.index), v[0]           # v(empty set) = the baseline

background = X_tr.sample(100, random_state=0)
phi, baseline = shapley_exact(gbm, x, background)
print(phi.sort_values(key=abs, ascending=False).round(1).to_string())
total = baseline + phi.sum()
print(f"baseline {baseline:.1f} + contributions {phi.sum():.1f} = {total:.1f}")
print(f"model prediction {gbm.predict(x.to_frame().T)[0]:.1f}")

Optional: the `shap` library's TreeSHAP gives the same numbers, exactly and fast. This cell installs `shap` and is skipped by the course checker; nothing below depends on it.

In [ ]:
# Downloads a model or data on first run
%pip install -q shap
import shap

# TreeSHAP with the same background data reproduces the exact values above
explainer = shap.TreeExplainer(gbm, data=background,
                               feature_perturbation="interventional")
sv = explainer(x.to_frame().T)
print(pd.Series(sv.values[0], index=X.columns)
      .sort_values(key=abs, ascending=False).round(1).to_string())
print("baseline:", sv.base_values)

### Hands-on: Shapley values by sampling

The definition is an average over orderings, so you can estimate it by sampling orderings.

1. Write `shapley_sampled(model, x, background, n_perm)`. For each random ordering, start from a random background patient, switch the features to this patient's values one at a time in that order, and credit each feature with the change in prediction when it switched.
2. Average the credits and compare with the exact `phi` for 10, 100, 1,000 and 10,000 orderings. Count the model calls each needs.

Expect the largest error to fall from about ten points to well under one.

In [ ]:
# Your code here

### Counterfactual explanations

The smallest change to the input that changes the outcome. For illustration only, suppose patients predicted above 200 are flagged; how little would have to change for this patient not to be?

In [ ]:
def counterfactual(model, x, features, target, X_ref, steps=25):
    """The closest change to `features` that brings the prediction below target."""
    grids = [np.linspace(X_ref[f].min(), X_ref[f].max(), steps) for f in features]
    n = steps ** len(features)
    cands = pd.DataFrame(np.tile(x.to_numpy(), (n, 1)), columns=x.index)
    cands[features] = np.array(list(product(*grids)))   # stay in the observed range
    ok = model.predict(cands) < target
    dist = (np.abs(cands[features] - x[features]) / X_ref[features].std()).sum(axis=1)
    return cands.loc[dist[ok].idxmin()]

for features in [["bmi", "bp"], ["bmi", "bp", "s5"]]:
    cf = counterfactual(gbm, x, features, target=200, X_ref=X_tr)
    change = ", ".join(f"{f} {x[f]:.1f} -> {cf[f]:.1f}" for f in features)
    print(f"{change}: prediction {gbm.predict(cf.to_frame().T)[0]:.0f}")

## Explaining neural networks

### Saliency maps and integrated gradients

$$
\mathrm{IG}_i(\mathbf{x}) = (x_i - x'_i)\int_0^1 \frac{\partial f\big(\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}')\big)}{\partial x_i}\,d\alpha
$$

Completeness: the attributions sum to $f(\mathbf{x}) - f(\mathbf{x}')$. We train a small CNN on the 8x8 digits, then check completeness.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits

torch.set_num_threads(1)     # tiny networks: extra threads only add overhead
digits = load_digits()
images = torch.tensor(digits.images / 16, dtype=torch.float32)  # pixels in [0, 1]
images = images.unsqueeze(1)                                    # (1797, 1, 8, 8)
labels = torch.tensor(digits.target)
g = torch.Generator().manual_seed(0)
order = torch.randperm(len(labels), generator=g)
train_idx, test_idx = order[:1400], order[1400:]

def make_cnn(seed):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # (32, 4, 4)
        nn.Flatten(), nn.Linear(32 * 4 * 4, 64), nn.ReLU(), nn.Linear(64, 10))

cnn = make_cnn(seed=0)
opt = torch.optim.Adam(cnn.parameters(), lr=2e-3)
for epoch in range(12):
    for batch in torch.randperm(1400, generator=g).split(64):
        b = train_idx[batch]
        loss = F.cross_entropy(cnn(images[b]), labels[b])
        opt.zero_grad(); loss.backward(); opt.step()
cnn.eval()
with torch.no_grad():
    acc = (cnn(images[test_idx]).argmax(1) == labels[test_idx]).float().mean()
print(f"test accuracy {acc:.3f}")

def saliency(model, img, c):
    """|d score_c / d pixel|: how sensitive the class score is to each pixel."""
    img = img.clone().requires_grad_(True)
    model(img[None])[0, c].backward()
    return img.grad[0].abs()

def integrated_gradients(model, img, c, baseline=None, steps=64):
    """(x - x') times the gradient averaged along the straight path from x' to x."""
    if baseline is None:
        baseline = torch.zeros_like(img)                   # a black image
    alphas = ((torch.arange(steps) + 0.5) / steps).view(-1, 1, 1, 1)
    path = (baseline + alphas * (img - baseline)).requires_grad_(True)
    model(path)[:, c].sum().backward()
    return ((img - baseline) * path.grad.mean(0))[0]

img, c = images[test_idx[0]], labels[test_idx[0]].item()
ig = integrated_gradients(cnn, img, c)
with torch.no_grad():
    gap = cnn(img[None])[0, c] - cnn(torch.zeros_like(img)[None])[0, c]
print(f"digit {c}: sum of IG {ig.sum():.3f}, f(x) - f(baseline) {gap:.3f}")

Saliency and integrated gradients for a few test digits.

In [ ]:
# Each row: a test digit, its saliency map and its integrated-gradients map.
# Red = pushes the class score up, blue = pushes it down.
fig, axes = plt.subplots(3, 4, figsize=(9, 6.5))
for col, i in enumerate(test_idx[:4]):
    im, cl = images[i], labels[i].item()
    sal_map = saliency(cnn, im, cl)
    ig_map = integrated_gradients(cnn, im, cl)
    lim = ig_map.abs().max()
    axes[0, col].imshow(im[0], cmap="gray_r")
    axes[0, col].set_title(f"digit {cl}")
    axes[1, col].imshow(sal_map, cmap="viridis")
    axes[2, col].imshow(ig_map, cmap="RdBu_r", vmin=-lim, vmax=lim)
for row, name in enumerate(["input", "saliency", "integrated gradients"]):
    axes[row, 0].set_ylabel(name)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### Do the explanations explain the model?

Two tests: the randomisation sanity check (Adebayo et al., 2018) and a deletion curve for faithfulness, with two random baselines.

In [ ]:
def rank_corr(a, b):
    """Spearman correlation: do two maps rank the pixels in the same order?"""
    ra, rb = (t.flatten().argsort().argsort().float() for t in (a, b))
    return torch.corrcoef(torch.stack([ra, rb]))[0, 1].item()

# 1. Sanity check: explain an untrained network. A faithful map should change.
random_cnn = make_cnn(seed=1)                  # same architecture, untrained
rows = []
for i in test_idx[:100]:
    im, cl = images[i], labels[i].item()
    ig_rand = integrated_gradients(random_cnn, im, cl).abs()
    rows.append([rank_corr(saliency(cnn, im, cl), saliency(random_cnn, im, cl)),
                 rank_corr(integrated_gradients(cnn, im, cl).abs(), ig_rand),
                 rank_corr(ig_rand, im[0])])
r = np.mean(rows, axis=0)
print(f"trained vs random model: saliency {r[0]:.2f}, IG {r[1]:.2f}")
print(f"IG of the random model vs the input image itself: {r[2]:.2f}")

# 2. Faithfulness: black out the most-attributed pixels first and watch P(class) fall
def deletion_score(model, img, c, attribution):
    rank = attribution.flatten().argsort(descending=True).argsort()   # 0 = top pixel
    ks = torch.arange(0, 65, 4)                                       # pixels removed
    batch = img.flatten() * (rank[None] >= ks[:, None])               # (17, 64)
    with torch.no_grad():
        probs = model(batch.view(-1, 1, 8, 8)).softmax(-1)[:, c]
    return probs.mean().item()          # area under the curve: lower = more faithful

gen = torch.Generator().manual_seed(0)
scores = {"integrated gradients": [], "saliency": [],
          "random pixels": [], "random ink pixels": []}
for i in test_idx[:200]:
    im, cl = images[i], labels[i].item()
    noise = torch.rand(8, 8, generator=gen)
    attributions = {"integrated gradients": integrated_gradients(cnn, im, cl),
                    "saliency": saliency(cnn, im, cl),
                    "random pixels": noise,
                    "random ink pixels": (im[0] > 0).float() + 0.5 * noise}
    for name, a in attributions.items():
        scores[name].append(deletion_score(cnn, im, cl, a))
for name, s in scores.items():
    print(f"{name:21s} deletion score {np.mean(s):.3f}")

## Looking inside a transformer

### A model small enough to read

A segment of distinct tokens, the same segment again, then filler. Predicting the second copy needs an induction mechanism. Two attention-only layers, two heads each, no MLPs, no LayerNorm. Training takes seconds.

In [ ]:
VOCAB, T = 24, 24

def make_batch(n, gen, L=None):
    """A segment of L distinct tokens, the same segment again, then random filler."""
    if L is None:
        L = torch.randint(4, 13, (n, 1), generator=gen)   # varies per sequence
    else:
        L = torch.full((n, 1), L)
    seg = torch.rand(n, VOCAB, generator=gen).argsort(1)[:, :12]  # distinct tokens
    pos = torch.arange(T)
    src = torch.where(pos < L, pos, pos - L).clamp(max=11)   # index into seg
    filler = torch.randint(0, VOCAB, (n, T), generator=gen)
    seq = torch.where(pos < 2 * L, seg.gather(1, src), filler)
    predictable = (pos[:-1] >= L) & (pos[:-1] <= 2 * L - 2)  # next token repeats
    return seq, predictable

class Attention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.dh = n_heads, d // n_heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).unbind(2)  # (B,T,h,dh)
        scores = torch.einsum("bqhd,bkhd->bhqk", q, k) / self.dh ** 0.5
        future = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        self.pattern = scores.masked_fill(future, float("-inf")).softmax(-1)  # kept
        self.z = torch.einsum("bhqk,bkhd->bqhd", self.pattern, v)   # per head
        return self.out(self.z.reshape(B, T, D))

class TinyTransformer(nn.Module):
    """Two attention-only layers, no MLPs and no LayerNorm: small enough to read."""
    def __init__(self, d=32, n_heads=2, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(T, d)
        self.layers = nn.ModuleList(Attention(d, n_heads) for _ in range(n_layers))
        self.unembed = nn.Linear(d, VOCAB, bias=False)

    def forward(self, tokens, edit=None):
        x = self.embed(tokens) + self.pos(torch.arange(tokens.shape[1]))
        resid = []               # residual stream before each layer and at the end
        for i in range(len(self.layers) + 1):
            if edit is not None:
                x = edit(i, x)           # a hook for interventions
            resid.append(x)
            if i < len(self.layers):
                x = x + self.layers[i](x)
        return self.unembed(x), resid

torch.manual_seed(0)
gen = torch.Generator().manual_seed(0)
model = TinyTransformer()
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(601):
    seq, m = make_batch(64, gen)
    logits, _ = model(seq)
    pred, target = logits[:, :-1][m], seq[:, 1:][m]
    loss = F.cross_entropy(pred, target)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 150 == 0:
        acc = (pred.argmax(-1) == target).float().mean()
        print(f"step {step:3d}  loss {loss.item():.3f}  "
              f"accuracy on repeats {acc:.2f}")

### Attention patterns, and what they do not tell you

Score each head as a previous-token head and as an induction head, then measure each component's direct contribution to the correct logit.

In [ ]:
L = 8
seq, m = make_batch(500, torch.Generator().manual_seed(1), L=L)
with torch.no_grad():
    logits, resid = model(seq)
t = torch.arange(1, L)                 # positions in the first copy
q = torch.arange(L, 2 * L - 1)         # positions in the second copy
for i, layer in enumerate(model.layers):
    for h in range(layer.h):
        P = layer.pattern[:, h]                        # (batch, query, key)
        prev = P[:, t, t - 1].mean()       # attention to the previous token
        ind = P[:, q, q - L + 1].mean()    # to the token after the earlier copy
        print(f"L{i}H{h}: previous-token score {prev:.2f}, induction score {ind:.2f}")

# Direct logit attribution: with no LayerNorm, logits = W_U (sum of everything written
# into the residual stream), so each component's share of the correct logit is exact.
def head_outputs(layer):
    W = layer.out.weight.view(-1, layer.h, layer.dh)    # (d, h, dh)
    return torch.einsum("bthe,dhe->bthd", layer.z, W)   # (B, T, h, d)

target = seq[:, 1:]
with torch.no_grad():
    parts = {"embed+pos": resid[0]}
    for i, layer in enumerate(model.layers):
        for h, out in enumerate(head_outputs(layer).unbind(2)):
            parts[f"L{i}H{h}"] = out
    for name, part in parts.items():
        lg = part[:, :-1] @ model.unembed.weight.T   # direct effect on the logits
        share = lg.gather(-1, target[..., None])[..., 0] - lg.mean(-1)
        print(f"{name:9s} adds {share[m].mean():5.2f} to the correct logit")

Attention patterns for one sequence: L0H0 attends one step back; the layer-1 heads attend to the token after the earlier copy.

In [ ]:
# Attention patterns for one sequence. Rows are query positions, columns are keys.
s0 = seq[:1]
with torch.no_grad():
    model(s0)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (i, h) in zip(axes, [(0, 0), (0, 1), (1, 0), (1, 1)]):
    ax.imshow(model.layers[i].pattern[0, h, :2 * L, :2 * L], cmap="Blues")
    ax.set_title(f"layer {i}, head {h}")
    ax.set_xlabel("key position")
axes[0].set_ylabel("query position")
plt.tight_layout()
plt.show()
print("tokens:", s0[0, :2 * L].tolist())

### Linear probes and the logit lens

Can a linear probe read the *previous* token from the residual stream at each depth? The untrained network is the control. Then read each point of the stream with the model's own unembedding.

In [ ]:
def residuals(m, n, seed):
    s, _ = make_batch(n, torch.Generator().manual_seed(seed))
    with torch.no_grad():
        _, r = m(s)
    return s, r

def probe_accuracy(acts_tr, y_tr, acts_te, y_te, steps=200):
    """Train a linear classifier on frozen activations; report held-out accuracy."""
    torch.manual_seed(0)
    probe = nn.Linear(acts_tr.shape[-1], VOCAB)
    opt = torch.optim.Adam(probe.parameters(), lr=0.05)
    for _ in range(steps):
        loss = F.cross_entropy(probe(acts_tr), y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
    return (probe(acts_te).argmax(-1) == y_te).float().mean().item()

torch.manual_seed(1)
untrained = TinyTransformer()          # control: same shape, random weights
for name, net in [("trained", model), ("untrained", untrained)]:
    s_tr, r_tr = residuals(net, 1000, seed=2)
    s_te, r_te = residuals(net, 500, seed=3)
    accs = []
    for i in range(3):                               # resid[0], resid[1], resid[2]
        # at positions 1-11, can a linear probe read off the PREVIOUS token?
        a_tr = r_tr[i][:, 1:12].reshape(-1, 32)
        a_te = r_te[i][:, 1:12].reshape(-1, 32)
        accs.append(probe_accuracy(a_tr, s_tr[:, :11].reshape(-1),
                                   a_te, s_te[:, :11].reshape(-1)))
    print(f"{name:9s} previous-token probe accuracy:", np.round(accs, 3))

# The logit lens: read each point of the stream with the model's own unembedding
for i, point in enumerate(resid):
    with torch.no_grad():
        read = model.unembed(point)[:, :-1][m].argmax(-1)
    acc = (read == target[m]).float().mean()
    print(f"logit lens at resid[{i}]: correct next token {acc:.2f}")

### Activation patching: from correlation to cause

Clean and corrupted runs differ only in the token at position 4. Patch one clean residual-stream vector at a time into the corrupted run and measure how much of the clean answer comes back at position 11.

In [ ]:
L, k = 8, 4
q = L + k - 1                  # this position holds the token seg[k-1] again
clean, _ = make_batch(200, torch.Generator().manual_seed(4), L=L)
answer = clean[:, k]           # so the model should predict seg[k]
# corrupt: replace token k in the first copy only, with a token not in the segment
unused = torch.ones(200, VOCAB).scatter(1, clean[:, :L], 0.0)
wrong = torch.multinomial(unused, 1, generator=torch.Generator().manual_seed(5))[:, 0]
corrupt = clean.clone()
corrupt[:, k] = wrong

def logit_diff(logits):
    lg = logits[:, q]
    diff = lg.gather(1, answer[:, None]) - lg.gather(1, wrong[:, None])
    return diff.mean().item()

with torch.no_grad():
    clean_logits, clean_resid = model(clean)
    corrupt_logits, _ = model(corrupt)
ld_clean, ld_corrupt = logit_diff(clean_logits), logit_diff(corrupt_logits)
print(f"logit difference: clean {ld_clean:.1f}, corrupted {ld_corrupt:.1f}")

recovered = np.zeros((3, q + 1))
for i in range(3):
    for p in range(q + 1):
        def patch(layer_i, x, i=i, p=p):    # copy one clean vector into the run
            if layer_i == i:
                x = x.clone()
                x[:, p] = clean_resid[i][:, p]
            return x
        with torch.no_grad():
            patched, _ = model(corrupt, edit=patch)
        recovered[i, p] = (logit_diff(patched) - ld_corrupt) / (ld_clean - ld_corrupt)
print("position  ", "".join(f"{p:5d}" for p in range(q + 1)))
for i in range(3):
    print(f"resid[{i}]  ", "".join(f"{v:5.1f}" for v in recovered[i]))

In [ ]:
plt.figure(figsize=(7, 2.2))
plt.imshow(recovered, cmap="Blues", vmin=0, vmax=1, aspect="auto")
plt.yticks(range(3), ["resid[0]", "resid[1]", "resid[2]"])
plt.xticks(range(q + 1))
plt.xlabel("position patched")
plt.colorbar(label="fraction recovered")
plt.title("Activation patching: where the information lives")
plt.tight_layout()
plt.show()

### Hands-on: knock out heads

1. Write `ablate(heads)` that zeroes the columns of each layer's `out.weight` belonging to the listed heads, for example `[(0, 0)]`, measures accuracy on repeats over 500 fresh sequences, and then restores the weights.
2. Ablate each head alone, then both layer-1 heads together.
3. Compare with the direct logit attribution. Which heads are necessary, and which matter only together?

In [ ]:
# Your code here

### Superposition and sparse autoencoders

The stream entering layer 1 holds the current token, the previous token and the position (59 possible values at positions 1 to 11) in 32 dimensions. A sparse autoencoder with 128 features tries to pull them apart; we know the ground truth, so we can check its features.

In [ ]:
seqs, _ = make_batch(2000, torch.Generator().manual_seed(6))
with torch.no_grad():
    _, stream = model(seqs)
acts = stream[1][:, 1:12].reshape(-1, 32)   # entering layer 1, first copy
truth = {"current token": (seqs[:, 1:12].reshape(-1), VOCAB),   # what we can check
         "previous token": (seqs[:, :11].reshape(-1), VOCAB),
         "position": (torch.arange(1, 12).repeat(len(seqs)), T)}

class SparseAutoencoder(nn.Module):
    def __init__(self, d, n_features):
        super().__init__()
        self.enc = nn.Linear(d, n_features)
        self.dec = nn.Linear(n_features, d)

    def forward(self, x):
        f = F.relu(self.enc(x - self.dec.bias))     # sparse, non-negative activations
        return self.dec(f), f

torch.manual_seed(0)
sae = SparseAutoencoder(32, 128)                     # 4 times more features than dims
opt = torch.optim.Adam(sae.parameters(), lr=3e-3)
for step in range(2000):
    xb = acts[torch.randint(len(acts), (1024,))]
    x_hat, f = sae(xb)
    sparsity = (f * sae.dec.weight.norm(dim=0)).sum(-1).mean()   # L1 penalty
    loss = (x_hat - xb).pow(2).sum(-1).mean() + 1.0 * sparsity
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    x_hat, f = sae(acts)
explained = 1 - (x_hat - acts).pow(2).sum() / (acts - acts.mean(0)).pow(2).sum()
print(f"variance explained {explained:.3f}, features active per vector "
      f"{(f > 0).float().sum(1).mean():.1f} of 128")

def correlations(f, values, n):
    """Correlation of every feature with every one-hot indicator, e.g. 'token = 7'."""
    onehot = F.one_hot(values, n).float()
    fz = (f - f.mean(0)) / (f.std(0) + 1e-8)
    oz = (onehot - onehot.mean(0)) / (onehot.std(0) + 1e-8)
    return fz.T @ oz / len(f)                         # (features, n)

corr = {kind: correlations(f, v, n) for kind, (v, n) in truth.items()}
clean_counts, covered = {}, set()
for j in range(128):
    best_r, kind, value = max((c[j].max().item(), kind, c[j].argmax().item())
                              for kind, c in corr.items())
    if best_r > 0.8:
        clean_counts[kind] = clean_counts.get(kind, 0) + 1
        covered.add((kind, value))
print("features that track one variable (r > 0.8):", clean_counts)
print("ground-truth variables found:", len(covered), "of", 24 + 24 + 11)

### Steering

Add the SAE feature for "current token = b" at position 4, entering layer 1. If the account is right, the induction heads at position 11 should copy b.

In [ ]:
best = corr["current token"].argmax(0)     # the feature for 'current token = b'
typical = torch.stack([f[truth["current token"][0] == b, best[b]].mean()
                       for b in range(VOCAB)])        # its usual activation when on

L, k = 8, 4
q = L + k - 1
clean, _ = make_batch(300, torch.Generator().manual_seed(7), L=L)
unused = torch.ones(300, VOCAB).scatter(1, clean[:, :L], 0.0)
new = torch.multinomial(unused, 1, generator=torch.Generator().manual_seed(8))[:, 0]
direction = sae.dec.weight[:, best[new]].T * typical[new][:, None]   # (300, 32)
for alpha in [0, 1, 2, 4]:
    def steer(layer_i, x):
        if layer_i == 1:                    # entering layer 1, at position k
            x = x.clone()
            x[:, k] += alpha * direction
        return x
    with torch.no_grad():
        logits, _ = model(clean, edit=steer)
    pred = logits[:, q].argmax(-1)
    orig = (pred == clean[:, k]).float().mean()
    steered = (pred == new).float().mean()
    print(f"alpha {alpha}: predicts the original token {orig:.2f}, "
          f"the new one {steered:.2f}")

## What has been found in real models, and the limits

Optional: the logit lens on GPT-2 small. This cell downloads the model (about 500 MB) on first run and is skipped by the course checker; nothing depends on it. Watch the top guesses for the next token after "The Eiffel Tower is in the city of" change from block to block.

In [ ]:
# Downloads a model or data on first run
%pip install -q transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()
ids = tok("The Eiffel Tower is in the city of", return_tensors="pt").input_ids
with torch.no_grad():
    hidden = gpt2(ids, output_hidden_states=True).hidden_states  # 1 + 12 entries
    for i, h in enumerate(hidden):
        h = h[0, -1]                       # the last position predicts the next token
        if i < len(hidden) - 1:    # the last entry already has the final LayerNorm
            h = gpt2.transformer.ln_f(h)
        top = gpt2.lm_head(h).topk(3).indices
        label = f"after block {i - 1:2d}:" if i else "embeddings:    "
        print(label, [tok.decode(t) for t in top])

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/30-interpretability.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.